In [ ]:
#  I've decided to attempt to make proprietary optimization and backpropagation code.
#  The layer framework, however, is inspired by Sentdex's Neural Networks from Scratch YouTube playlist. Highly recommend it. 
#  This strategy of building this should give me the best practice in building neural networks, as the layer framework is quite simple and rather ubiquitous.

import sklearn as sk
import numpy as np
import pandas as pd

In [ ]:
#import training data


inputs = np.random.randn(shape = (32,32))

In [ ]:

#===========================================================================
#basic layer structure 
#===========================================================================


def loss_attr(class_name: str, attr_name: str):
    
    return getattr(class_name, attr_name)

#===========================================================================

class layer:        #initialize with MSE/MAE/LossFunctionAcronym as loss type, delta_ActivationFunction as act_class
    def __init__(self, order, n_neurons, n_inputs, act_class, loss_class, input):  #don't want self inputs as an attribute as it overcomplicates initialization of the class. inputs determined outside of the initialization. 
        self.loss = loss_class
        self.act = act_class
        self.init_weights = np.random.randn((n_neurons, n_inputs))      
        self.init_biases = np.zeros(1, n_neurons)
        self.order = order      #what number/position layer the layer at hand is
        self.input = input

    def forward_propagate(self):     
        self.output = self.input@self.init_weights + self.init_biases

class output_layer(layer):

    def dc_db(self):    #cost partial derivative with respect to bias
        delta = getattr(self.loss, self.act)
        self.dcdb = delta

    def dc_dw(self, prior_activ):
        delta = self.act.delta()
        self.dcdw = prior_activ*delta #not sure about correctness of dimensionality. just looking for a framework for now. prior_activ is inputs to the layer


class hidden_input_layer(layer):

    def dc_db(self):
        delta = getattr(self.loss, self.act)
        self.dcdb = delta

    def dc_dw(self):
        delta = getattr(self.loss, self.act)
        self.dcdb = delta


#===========================================================================
#activation functions
#===========================================================================

class ReLU:
    def __init__(self, loss):
        self.lossclass = loss

    def forward(self, input):
        self.output = max(0,input)

    def d_dz(self, input):
        if input <= 0:
            self.ddz = 0

        elif input > 0:
            self.ddz = 1

        else:
            print("Error in computation of ReLU derivative.")
            return 

    def delta(self):
        deltarelu = self.lossclass.dcda*ReLU.d_dz(self.inputs)
        self.deltarelu = deltarelu


class Sigmoid:
    def __init__(self, loss):
        self.lossclass = loss

    def forward(self, input):
        self.output = 1/(1+np.exp(-input))

    def d_dz(self, input):
        self.ddz = np.exp(-input)/((1+np.exp(-input))**2)

    def delta(self):
        deltasig = self.lossclass.dcda*Sigmoid.d_dz(self.inputs)
        self.deltasigmoid = deltasig
    

class Softmax:
    def __init__(self, loss):
        self.lossclass = loss

    def forward(self, input):
        numerator = np.exp(input, keepdims = True) #===> returns e^z_i for each value, changes matrix into these values. 
        denominator = np.sum(input, axis = 1, keepdims = True) #===> returns the sum over all values in a batch of samples, puts it into a matrix 
        self.output = numerator/denominator         #keepdims helps numpy broadcast properly. good to remember. 

    def d_dz():
        pass #to be added soon. 

    def delta(self):
        deltasoft = self.lossclass.dcda*Softmax.d_dz(self.inputs)
        self.deltasoftmax = deltasoft


#===========================================================================
#loss functions
#===========================================================================


class Loss:         #this parent class is trivial right now, but the point is: 
                   #if I update this someday, I will now not have to implement a new parent loss class.
   def mean(values):
        return np.mean(values)

class MeanSquaredError(Loss):
    def __init__(self, inputs, targets, layer_number):
        self.inputs = inputs
        self.targets = targets
        self.layer = layer_number

    def error(self):
        squared_summed = np.sum((self.targets-self.inputs)**2)
        self.output = self.mean(squared_summed)

    def dc_da(self):
            inner = 2*np.sum(self.targets-self.inputs)
            self.dcda = -1*(self.mean(inner))



class MeanAbsError(Loss):
    def __init__(self, inputs, targets):
        self.inputs = inputs
        self.targets = targets

    def error(self):
        abs_summed = np.sum(np.abs(self.inputs - self.targets))
        self.output = self.mean(abs_summed)

    def delta():
        pass







     
   

In [ ]:
#===========================================================================
#Implementation of Gradient Descent Algorithm
#===========================================================================

class optimizer:        
    def __init__(self, layer, loss, activation): 
        self.losstype = loss
        self.acttype = activation


class GradientDescent(optimizer):

    def optimize(self, epochs, initial_weights, initial_biases, learning_rate, n_layers):

        w = initial_weights
        b = initial_biases
       

        for i in range(epochs):

            for layer in range(n_layers):       #need to specify weights and biases for layers with slicing, then redo the matrix.

                #make dictionary with key pairs for layers to loop through them. 

                w_new = w - learning_rate * layer.dc_dw
                b_new = b - learning_rate * layer.dc_db

                w = w_new
                b = b_new


